# Lipschitz of Tanh (full autograd)

In [1]:
import torch
import torch.nn.functional as F
from torch.autograd.functional import jacobian as autograd_jacobian
import math


seed = 1
torch.manual_seed(seed)
ns = list(range(2, 16))      
steps = 10000                
lr = 1e-1                    
    
def Tanh(x): 
    return torch.tanh(x)


for n in ns:
    g = torch.Generator().manual_seed(seed)
    
    x = (0.01 * torch.randn(n, generator=g, dtype=torch.float64)).requires_grad_(True)
    
    optimizer = torch.optim.Adam([x], lr=lr)

    for _ in range(steps):
            
        J = autograd_jacobian(Tanh, x, create_graph=True)
                
        #K = torch.sqrt(torch.sum(J*J))   #<- it's ok but may have numeric error
        K = torch.linalg.norm(J, ord=2)   #<- more accurate              

        optimizer.zero_grad()
        (-K).backward()                                 
        optimizer.step()

    print(f"Tanh n = {n:2d}  Estimated Lipschitz constant = {K.item():.6f} at x = {x.detach().numpy()}")

    

Tanh n =  2  Estimated Lipschitz constant = 1.000000 at x = [-3.82049881e-11 -5.97878543e-01]
Tanh n =  3  Estimated Lipschitz constant = 1.000000 at x = [ 1.10547788e-05 -5.53217021e-01 -5.99921008e-01]
Tanh n =  4  Estimated Lipschitz constant = 1.000000 at x = [-7.41777377e-09 -5.53217021e-01 -5.99921008e-01 -5.28490885e-01]
Tanh n =  5  Estimated Lipschitz constant = 1.000000 at x = [ 9.97115050e-05 -5.53217021e-01 -5.99921008e-01 -5.16661624e-01
  5.30184475e-01]
Tanh n =  6  Estimated Lipschitz constant = 1.000000 at x = [ 2.35853984e-05 -5.32032961e-01 -5.99921008e-01 -5.09930845e-01
  5.18355193e-01  5.54223062e-01]
Tanh n =  7  Estimated Lipschitz constant = 1.000000 at x = [-5.06013614e-01 -5.32032961e-01 -5.99921008e-01 -5.09930845e-01
  5.18355193e-01  5.54223062e-01  2.38758558e-05]
Tanh n =  8  Estimated Lipschitz constant = 0.999472 at x = [-0.50470335 -0.53203296 -0.59992101 -0.50641387  0.51162439  0.55422306
 -0.02365181 -0.51905705]
Tanh n =  9  Estimated Lipschitz c